# Notebook 2 — Insider Threat Detector (UEBA)
## CyberAI SOC Platform — Multi-Source Temporal Behavioral Classifier

**Dataset:** CERT Insider Threat Dataset r4.2 — Carnegie Mellon University SEI  
**Source:** `andrihjonior/cert-insider-threat-dataset-r4-2` (Kaggle)  
**Detects:** `data_exfiltration`, `insider_sabotage`, `policy_violation`  
**Unit of analysis:** User-day behavioral vectors in 7-day sliding windows  
**Output ID format:** `UEBA-{user_id}-{date}`

---

### Architecture: 2-Phase Multi-Source Temporal Behavioral Classifier

```
Phase 1 — Feature Engineering (no neural network)
├── logon.csv  → daily_login_count, after_hours_logins, unique_PCs, weekend_logins
├── device.csv → usb_connects, after_hours_usb, new_device_flag
├── file.csv   → files_copied, sensitive_extensions, unique_filenames
├── http.csv   → unique_domains, after_hours_browsing, job_site_visits
└── email.csv  → emails_sent, external_recipients, attachments, after_hours_emails
     ↓
     Fused into ONE behavioral vector per user per day (~30 features)

Phase 2 — Temporal Deep Learning (7-day sliding windows)
     [Day1_vec, Day2_vec, ..., Day7_vec]
          ↓
     MultiScale Conv1D(3,5,7) + LayerNorm
          ↓
     BiLSTM(128, return_sequences=True)
          ↓
     MultiHeadAttention(8 heads) + Residual + LayerNorm
          ↓
     GlobalAvgPool → Dense(256,gelu) → Dense(128,gelu) → Softmax(fp32)
```

### Training stack:
- **Focal Loss** (α=0.25, γ=2.0) — handles extreme class imbalance (~99.9% normal)
- **Cosine Annealing Warm Restart** (T_0=10, T_mult=2)
- **Early stopping on macro F1** (patience=7)
- **Mixed precision fp16**
- **clipnorm=1.0**

## Step 1 — Environment Setup & Mixed Precision

In [4]:
%%time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, Bidirectional, LSTM,
    Dense, Dropout, Add, Concatenate, LayerNormalization,
    GlobalAveragePooling1D, MultiHeadAttention, BatchNormalization
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import joblib, json, gc, time, os, math, glob, warnings, datetime
warnings.filterwarnings('ignore')

mixed_precision.set_global_policy('mixed_float16')

OUTPUT_PATH = '/kaggle/working/'
DATA_DIR    = '/kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

WINDOW_SIZE  = 7
CHUNK_SIZE   = 500_000   # rows per chunk for large CSVs
SOURCE_NAME  = 'insider_threat'

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print(f'Mixed precision: {mixed_precision.global_policy().compute_dtype}')

# ── Auto-discover CSV files ──────────────────────────────────────────────────
all_files = glob.glob(os.path.join(DATA_DIR, '**', '*.csv'), recursive=True)
print(f'\nFound {len(all_files)} CSV files:')
for f in sorted(all_files):
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f'  {os.path.basename(f):30s} {size_mb:8.1f} MB')

2026-06-05 14:06:44.824909: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780668405.071512      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780668405.141481      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780668405.716918      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780668405.716966      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780668405.716968      58 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision: float16

Found 95 CSV files:
  insiders.csv                        0.0 MB
  r4.2-1-AAM0658.csv                  0.0 MB
  r4.2-1-AJR0932.csv                  0.0 MB
  r4.2-1-BDV0168.csv                  0.0 MB
  r4.2-1-BIH0745.csv                  0.0 MB
  r4.2-1-BLS0678.csv                  0.0 MB
  r4.2-1-BTL0226.csv                  0.0 MB
  r4.2-1-CAH0936.csv                  0.0 MB
  r4.2-1-DCH0843.csv                  0.0 MB
  r4.2-1-EHB0824.csv                  0.0 MB
  r4.2-1-EHD0584.csv                  0.0 MB
  r4.2-1-FMG0527.csv                  0.0 MB
  r4.2-1-FTM0406.csv                  0.0 MB
  r4.2-1-GHL0460.csv                  0.0 MB
  r4.2-1-HJB0742.csv                  0.0 MB
  r4.2-1-JMB0308.csv                  0.0 MB
  r4.2-1-JRG0207.csv                  0.0 MB
  r4.2-1-KLH0596.csv            

## Step 2 — Helper Classes (Focal Loss, Cosine Annealing, Macro-F1 Early Stopping)

In [5]:
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=0.25, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha; self.gamma = gamma
    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.squeeze(y_true), tf.int32)
        y_pred = tf.cast(tf.clip_by_value(y_pred, 1e-7, 1.0), tf.float32)
        y_oh   = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1], dtype=tf.float32)
        p_t    = tf.reduce_sum(y_oh * y_pred, axis=-1)
        return tf.reduce_mean(self.alpha * tf.pow(1.0 - p_t, self.gamma) * (-tf.math.log(p_t)))
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'alpha': self.alpha, 'gamma': self.gamma})
        return cfg

class CosineAnnealingWarmRestarts(tf.keras.callbacks.Callback):
    def __init__(self, initial_lr=0.001, T_0=10, T_mult=2, eta_min=1e-6):
        super().__init__()
        self.initial_lr = initial_lr; self.T_0 = T_0; self.T_mult = T_mult
        self.eta_min = eta_min; self.T_cur = 0; self.T_i = T_0
    def on_epoch_begin(self, epoch, logs=None):
        lr = self.eta_min + (self.initial_lr - self.eta_min) * \
             (1 + math.cos(math.pi * self.T_cur / self.T_i)) / 2
        tf.keras.backend.set_value(self.model.optimizer.learning_rate, lr)
        self.T_cur += 1
        if self.T_cur >= self.T_i:
            self.T_cur = 0; self.T_i = int(self.T_i * self.T_mult)

class MacroF1EarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, X_val, y_val, patience=7):
        super().__init__()
        self.X_val = X_val; self.y_val = y_val
        self.patience = patience; self.best_f1 = 0.0
        self.wait = 0; self.best_weights = None
    def on_epoch_end(self, epoch, logs=None):
        y_pred = np.argmax(self.model.predict(self.X_val, verbose=0), axis=1)
        f1 = f1_score(self.y_val, y_pred, average='macro', zero_division=0)
        print(f'  val_macro_f1: {f1:.4f} (best: {self.best_f1:.4f})')
        if f1 > self.best_f1:
            self.best_f1 = f1
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.set_weights(self.best_weights)
                self.model.stop_training = True
                print(f'  → Early stop (F1={self.best_f1:.4f})')

## Step 3 — Load Raw CSV Logs

We load each CSV independently to manage memory. Each file is parsed for its
specific columns and date formats.

In [17]:
def find_csv(name, data_dir=DATA_DIR):
    """Find a CSV file by name (case-insensitive) in the dataset directory."""
    candidates = glob.glob(os.path.join(data_dir, '**', f'*{name}*'), recursive=True)
    csv_files  = [c for c in candidates if c.endswith('.csv')]
    if csv_files:
        chosen = sorted(csv_files, key=lambda x: len(x))[0]
        size_mb = os.path.getsize(chosen) / (1024*1024)
        print(f'  Found {name}: {chosen} ({size_mb:.0f} MB)')
        return chosen
    print(f'  [WARN] {name}.csv not found')
    return None

def add_time_features(df, date_col='date'):
    """Parse date column and add time-based helper columns."""
    if date_col not in df.columns:
        for c in df.columns:
            if 'date' in c.lower() or 'time' in c.lower():
                date_col = c; break
    df['datetime'] = pd.to_datetime(df[date_col], errors='coerce', format='mixed')
    df = df.dropna(subset=['datetime'])
    df['day']  = df['datetime'].dt.normalize()
    df['hour'] = df['datetime'].dt.hour
    df['is_after_hours'] = ((df['hour'] < 8) | (df['hour'] >= 18)).astype(np.int8)
    df['is_weekend']     = (df['datetime'].dt.dayofweek >= 5).astype(np.int8)
    return df

def get_user_col(df):
    """Find the user column."""
    for c in ['user', 'user_id', 'userid', 'employee']:
        if c in df.columns: return c
    return df.columns[1] if len(df.columns) > 1 else df.columns[0]

print('✓ Utility functions ready.')

✓ Utility functions ready.


## Step 4 — Load Insider Threat Labels

The CERT r4.2 dataset includes an `answers/` directory or an `insiders.csv`
file that identifies which users are insiders and when their malicious
activity occurred.

In [18]:
%%time

insider_path = find_csv('insider') or find_csv('answer')
if insider_path is None:
    answer_dirs = glob.glob(os.path.join(DATA_DIR, '**', 'answers'), recursive=True)
    for d in answer_dirs:
        csvs = glob.glob(os.path.join(d, '*.csv'))
        if csvs: insider_path = csvs[0]; break

insider_users = {}

if insider_path:
    df_ins = pd.read_csv(insider_path, low_memory=False, on_bad_lines='skip')
    df_ins.columns = [c.lower().strip() for c in df_ins.columns]
    print(f'Insiders columns: {list(df_ins.columns)}')
    print(df_ins.head(10))

    user_col = next((c for c in ['user', 'user_id', 'userid'] if c in df_ins.columns), df_ins.columns[0])
    start_col = next((c for c in ['start', 'start_date'] if c in df_ins.columns), None)
    end_col   = next((c for c in ['end', 'end_date'] if c in df_ins.columns), None)
    scen_col  = next((c for c in ['scenario', 'type', 'category'] if c in df_ins.columns), None)

    for _, row in df_ins.iterrows():
        uid = str(row[user_col]).strip()
        start = pd.to_datetime(row[start_col], errors='coerce') if start_col else None
        end   = pd.to_datetime(row[end_col], errors='coerce') if end_col else None
        scen  = str(row[scen_col]).strip() if scen_col else 'insider_threat'
        insider_users.setdefault(uid, []).append({'start': start, 'end': end, 'scenario': scen})
    del df_ins
else:
    print('[WARN] No insiders/answers file found.')

print(f'\n✓ {len(insider_users)} insider users identified')
for uid, ev in list(insider_users.items())[:5]:
    print(f'  {uid}: scenario={ev[0]["scenario"]}')

  Found insider: /kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/answers/insiders.csv (0 MB)
Insiders columns: ['dataset', 'scenario', 'details', 'user', 'start', 'end']
   dataset  scenario             details     user                start  \
0      2.0         1              r2.csv  ONS0995     3/6/2010 1:41:56   
1      3.1         1          r3.1-1.csv  CSF0929  07/01/2010 01:24:58   
2      3.1         2          r3.1-2.csv  CCH0959  08/02/2010 10:34:31   
3      3.2         1          r3.2-1.csv  RCW0822  09/29/2010 21:10:27   
4      3.2         2          r3.2-2.csv  JCE0258  07/12/2010 08:16:02   
5      4.1         1          r4.1-1.csv  ABB0427  08/21/2010 05:20:34   
6      4.1         2          r4.1-2.csv  HFC0492  08/23/2010 10:40:56   
7      4.1         3          r4.1-3.csv  KTW0365  07/08/2010 11:51:00   
8      4.2         1  r4.2-1-AAM0658.csv  AAM0658  10/23/2010 01:34:19   
9      4.2         1  r4.2-1-AJR0932.csv  AJR0932  09/10/2010 19:12:0

## Step 5 — Phase 1: Multi-Source Behavioral Feature Engineering

For each user, for each day, we extract behavioral features from all 5 log
sources and fuse them into a single feature vector. This is the core of
what enterprise UEBA products (Splunk UEBA, Exabeam, Microsoft Sentinel) do.

In [19]:
%%time

logon_features = pd.DataFrame()
logon_path = find_csv('logon')

if logon_path:
    df = pd.read_csv(logon_path, low_memory=False, on_bad_lines='skip')
    df.columns = [c.lower().strip() for c in df.columns]
    print(f'  Loaded {len(df):,} rows. Columns: {list(df.columns)}')
    df = add_time_features(df)
    user_col = get_user_col(df)

    grp = df.groupby([user_col, 'day'])
    logon_features = grp.agg(
        logon_total   = ('datetime', 'count'),
        logon_after_h = ('is_after_hours', 'sum'),
        logon_weekend = ('is_weekend', 'sum'),
    ).reset_index()

    if 'pc' in df.columns:
        pcs = grp['pc'].nunique().reset_index(name='logon_unique_pcs')
        logon_features = logon_features.merge(pcs, on=[user_col, 'day'], how='left')
    else:
        logon_features['logon_unique_pcs'] = 1

    act_col = 'activity' if 'activity' in df.columns else None
    if act_col:
        act = df[act_col].str.lower()
        df['_on']  = act.str.contains('logon|login', na=False).astype(np.int8)
        df['_off'] = act.str.contains('logoff|logout', na=False).astype(np.int8)
        onoff = df.groupby([user_col, 'day']).agg(
            logon_count=('_on','sum'), logoff_count=('_off','sum')).reset_index()
        logon_features = logon_features.merge(onoff, on=[user_col, 'day'], how='left')
    else:
        logon_features['logon_count']  = logon_features['logon_total']
        logon_features['logoff_count'] = 0

    logon_features.rename(columns={user_col: 'user'}, inplace=True)
    print(f'✓ Logon features: {logon_features.shape}')
    del df; gc.collect()
else:
    print('[SKIP] No logon data')

print(f'Memory after logon: ~{gc.collect()} objects freed')

  Found logon: /kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/logon.csv (56 MB)
  Loaded 854,859 rows. Columns: ['id', 'date', 'user', 'pc', 'activity']
✓ Logon features: (330452, 8)
Memory after logon: ~0 objects freed
CPU times: user 47.6 s, sys: 189 ms, total: 47.8 s
Wall time: 47.8 s


In [20]:
%%time

device_features = pd.DataFrame()
device_path = find_csv('device')

if device_path:
    df = pd.read_csv(device_path, low_memory=False, on_bad_lines='skip')
    df.columns = [c.lower().strip() for c in df.columns]
    print(f'  Loaded {len(df):,} rows. Columns: {list(df.columns)}')
    df = add_time_features(df)
    user_col = get_user_col(df)

    grp = df.groupby([user_col, 'day'])
    device_features = grp.agg(
        device_total   = ('datetime', 'count'),
        device_after_h = ('is_after_hours', 'sum'),
        device_weekend = ('is_weekend', 'sum'),
    ).reset_index()

    act_col = 'activity' if 'activity' in df.columns else None
    if act_col:
        df['_conn'] = df[act_col].str.lower().str.contains('connect', na=False).astype(np.int8)
        conn = df.groupby([user_col, 'day'])['_conn'].sum().reset_index(name='device_connects')
        device_features = device_features.merge(conn, on=[user_col, 'day'], how='left')
    else:
        device_features['device_connects'] = device_features['device_total']

    device_features.rename(columns={user_col: 'user'}, inplace=True)
    print(f'✓ Device features: {device_features.shape}')
    del df; gc.collect()
else:
    print('[SKIP] No device data')

  Found device: /kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/device.csv (28 MB)
  Loaded 405,380 rows. Columns: ['id', 'date', 'user', 'pc', 'activity']
✓ Device features: (55717, 6)
CPU times: user 22.1 s, sys: 104 ms, total: 22.2 s
Wall time: 22.2 s


In [21]:
%%time

file_features = pd.DataFrame()
file_path_csv = find_csv('file')

if file_path_csv:
    # Only load columns we need (skip 'content' which is huge)
    try:
        peek = pd.read_csv(file_path_csv, nrows=2, low_memory=False)
        peek.columns = [c.lower().strip() for c in peek.columns]
        needed = [c for c in peek.columns if c not in ('content',)]
        print(f'  File columns available: {list(peek.columns)}')
        print(f'  Loading only: {needed}')
        del peek
    except:
        needed = None

    df = pd.read_csv(file_path_csv, usecols=needed, low_memory=False, on_bad_lines='skip')
    df.columns = [c.lower().strip() for c in df.columns]
    print(f'  Loaded {len(df):,} rows')
    df = add_time_features(df)
    user_col = get_user_col(df)

    grp = df.groupby([user_col, 'day'])
    file_features = grp.agg(
        file_total    = ('datetime', 'count'),
        file_after_h  = ('is_after_hours', 'sum'),
        file_weekend  = ('is_weekend', 'sum'),
    ).reset_index()

    fname_col = next((c for c in ['filename','file','filepath','name'] if c in df.columns), None)
    if fname_col:
        n_unique = grp[fname_col].nunique().reset_index(name='file_unique_names')
        file_features = file_features.merge(n_unique, on=[user_col, 'day'], how='left')

        sensitive = ('.exe','.zip','.rar','.7z','.doc','.docx','.pdf','.key','.pem','.xls','.xlsx')
        df['_sens'] = df[fname_col].apply(
            lambda x: int(str(x).lower().endswith(sensitive)) if pd.notna(x) else 0
        ).astype(np.int8)
        sens = df.groupby([user_col, 'day'])['_sens'].sum().reset_index(name='file_sensitive')
        file_features = file_features.merge(sens, on=[user_col, 'day'], how='left')
    else:
        file_features['file_unique_names'] = 0
        file_features['file_sensitive']    = 0

    file_features.rename(columns={user_col: 'user'}, inplace=True)
    print(f'✓ File features: {file_features.shape}')
    del df; gc.collect()
else:
    print('[SKIP] No file data')

  Found file: /kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/file.csv (184 MB)
  File columns available: ['id', 'date', 'user', 'pc', 'filename', 'content']
  Loading only: ['id', 'date', 'user', 'pc', 'filename']
  Loaded 445,581 rows
✓ File features: (45907, 7)
CPU times: user 26.2 s, sys: 201 ms, total: 26.4 s
Wall time: 26.5 s


In [ ]:
%%time

HTTP_CACHE = OUTPUT_PATH + 'http_features_cache.parquet'

# ── LOAD CACHE if it exists ──────────────────────────────────────────────────
if os.path.exists(HTTP_CACHE):
    http_features = pd.read_parquet(HTTP_CACHE)
    http_features['day'] = pd.to_datetime(http_features['day'])
    print(f'✓ Loaded HTTP features from cache: {http_features.shape}')
else:
    # ── COMPUTE ───────────────────────────────────────────────────────────────
    http_features = pd.DataFrame()
    http_path = find_csv('http')

    if http_path:
        try:
            peek = pd.read_csv(http_path, nrows=2, low_memory=False)
            peek.columns = [c.lower().strip() for c in peek.columns]
            needed = [c for c in peek.columns if c != 'content']
            print(f'  HTTP columns: {list(peek.columns)}')
            print(f'  Loading only: {needed}')
            del peek
        except Exception as e:
            print(f'  [WARN] peek failed: {e} — loading all columns')
            needed = None

        job_sites = ['indeed','linkedin','monster','glassdoor','careerbuilder',
                     'ziprecruiter','simplyhired','job','career','resume',
                     'wikileaks','dropbox','mega.nz','pastebin']

        chunk_features = []
        url_records    = []
        chunk_num = 0

        for chunk in pd.read_csv(http_path, usecols=needed, chunksize=CHUNK_SIZE,
                                 low_memory=False, on_bad_lines='skip'):
            chunk.columns = [c.lower().strip() for c in chunk.columns]
            chunk = add_time_features(chunk)
            user_col = get_user_col(chunk)
            url_col  = next((c for c in ['url','website','domain','uri'] if c in chunk.columns), None)

            grp = chunk.groupby([user_col, 'day'])
            feat = grp.agg(
                http_total   = ('datetime', 'count'),
                http_after_h = ('is_after_hours', 'sum'),
                http_weekend = ('is_weekend', 'sum'),
            ).reset_index()

            if url_col:
                raw = chunk[[user_col, 'day', url_col]].copy()
                raw.columns = ['user', 'day', '_url']
                url_records.append(raw)

                chunk['_susp'] = chunk[url_col].apply(
                    lambda x: int(any(s in str(x).lower() for s in job_sites)) if pd.notna(x) else 0
                ).astype(np.int16)
                susp = chunk.groupby([user_col, 'day'])['_susp'].sum().reset_index(name='http_suspicious')
                feat = feat.merge(susp, on=[user_col, 'day'], how='left')
            else:
                feat['http_suspicious'] = 0

            feat['http_unique_domains'] = 0
            feat.rename(columns={user_col: 'user'}, inplace=True)
            chunk_features.append(feat)
            chunk_num += 1
            print(f'    Chunk {chunk_num}: {len(chunk):,} rows → {len(feat):,} user-days')
            del chunk; gc.collect()

        if chunk_features:
            http_features = pd.concat(chunk_features, ignore_index=True)
            del chunk_features

            sum_cols = [c for c in http_features.columns
                        if c not in ('user', 'day', 'http_unique_domains')]
            http_features = (
                http_features
                .groupby(['user', 'day'])[sum_cols]
                .sum()
                .reset_index()
            )

            if url_records:
                all_urls = pd.concat(url_records, ignore_index=True)
                del url_records
                true_unique = (
                    all_urls
                    .groupby(['user', 'day'])['_url']
                    .nunique()
                    .reset_index(name='http_unique_domains')
                )
                del all_urls
                http_features = http_features.merge(true_unique, on=['user', 'day'], how='left')
                http_features['http_unique_domains'] = http_features['http_unique_domains'].fillna(0)
                del true_unique
            else:
                http_features['http_unique_domains'] = 0

            gc.collect()
            print(f'✓ HTTP features computed: {http_features.shape}')

    else:
        print('[SKIP] No HTTP data')

    # ── SAVE CACHE ────────────────────────────────────────────────────────────
    if not http_features.empty:
        http_features['day'] = pd.to_datetime(http_features['day'])
        http_features.to_parquet(HTTP_CACHE, index=False)
        print(f'✓ HTTP features cached → {HTTP_CACHE}')

  Found http: /kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/http.csv (13863 MB)
  HTTP columns: ['id', 'date', 'user', 'pc', 'url', 'content']
  Loading only: ['id', 'date', 'user', 'pc', 'url']
    Chunk 1: 500,000 rows → 6,208 user-days
    Chunk 2: 500,000 rows → 7,095 user-days
    Chunk 3: 500,000 rows → 6,317 user-days
    Chunk 4: 500,000 rows → 6,095 user-days
    Chunk 5: 500,000 rows → 7,061 user-days
    Chunk 6: 500,000 rows → 7,010 user-days
    Chunk 7: 500,000 rows → 6,029 user-days
    Chunk 8: 500,000 rows → 6,484 user-days
    Chunk 9: 500,000 rows → 6,998 user-days


In [4]:
%%time

email_features = pd.DataFrame()
email_path = find_csv('email')

if email_path:
    # Discover columns (skip 'content' — it's the email body text, very large)
    try:
        peek = pd.read_csv(email_path, nrows=2, low_memory=False)
        peek.columns = [c.lower().strip() for c in peek.columns]
        needed = [c for c in peek.columns if c not in ('content',)]
        print(f'  Email columns: {list(peek.columns)}')
        print(f'  Loading only: {needed}')
        del peek
    except:
        needed = None

    chunk_features = []
    chunk_num = 0
    for chunk in pd.read_csv(email_path, usecols=needed, chunksize=CHUNK_SIZE,
                             low_memory=False, on_bad_lines='skip'):
        chunk.columns = [c.lower().strip() for c in chunk.columns]
        chunk = add_time_features(chunk)
        user_col = get_user_col(chunk)

        grp = chunk.groupby([user_col, 'day'])

        agg_dict = {
            'email_total':   ('datetime', 'count'),
            'email_after_h': ('is_after_hours', 'sum'),
            'email_weekend': ('is_weekend', 'sum'),
        }

        attach_col = next((c for c in ['attachments','attachment'] if c in chunk.columns), None)
        if attach_col:
            chunk[attach_col] = pd.to_numeric(chunk[attach_col], errors='coerce').fillna(0)
            agg_dict['email_attachments'] = (attach_col, 'sum')

        size_col = next((c for c in ['size','email_size','bytes'] if c in chunk.columns), None)
        if size_col:
            chunk[size_col] = pd.to_numeric(chunk[size_col], errors='coerce').fillna(0)
            agg_dict['email_avg_size'] = (size_col, 'mean')

        feat = grp.agg(**agg_dict).reset_index()

        to_col = next((c for c in ['to','recipient','recipients'] if c in chunk.columns), None)
        if to_col:
            chunk['_nrecip'] = chunk[to_col].apply(
                lambda x: len(str(x).split(';')) if pd.notna(x) else 0
            )
            recip = chunk.groupby([user_col, 'day'])['_nrecip'].sum().reset_index(name='email_recipients')
            feat = feat.merge(recip, on=[user_col, 'day'], how='left')
        else:
            feat['email_recipients'] = 0

        for col in ['email_attachments', 'email_avg_size']:
            if col not in feat.columns:
                feat[col] = 0

        feat.rename(columns={user_col: 'user'}, inplace=True)
        chunk_features.append(feat)
        chunk_num += 1
        print(f'    Chunk {chunk_num}: {len(chunk):,} rows → {len(feat):,} user-days')
        del chunk; gc.collect()

    if chunk_features:
        email_features = pd.concat(chunk_features, ignore_index=True)
        num_cols = [c for c in email_features.columns if c not in ('user', 'day')]
        # For avg_size, we need mean not sum when re-aggregating
        agg_map = {c: 'sum' for c in num_cols}
        if 'email_avg_size' in agg_map:
            agg_map['email_avg_size'] = 'mean'
        email_features = email_features.groupby(['user', 'day']).agg(agg_map).reset_index()
        print(f'✓ Email features: {email_features.shape}')
        del chunk_features; gc.collect()
else:
    print('[SKIP] No email data')

NameError: name 'find_csv' is not defined

In [11]:
%%time

all_dfs = []
for name, df in [('logon', logon_features), ('device', device_features),
                 ('file', file_features), ('http', http_features),
                 ('email', email_features)]:
    if not df.empty and 'user' in df.columns and 'day' in df.columns:
        all_dfs.append(df)
        print(f'  Merging {name}: {df.shape}')

if len(all_dfs) == 0:
    raise RuntimeError('No feature tables to merge!')

df_daily = all_dfs[0]
for df_other in all_dfs[1:]:
    df_daily = df_daily.merge(df_other, on=['user', 'day'], how='outer')

feature_cols = [c for c in df_daily.columns if c not in ('user', 'day')]
df_daily[feature_cols] = df_daily[feature_cols].fillna(0).astype(np.float32)

df_daily['day'] = pd.to_datetime(df_daily['day'])
df_daily = df_daily.sort_values(['user', 'day']).reset_index(drop=True)

# Free individual feature tables
del logon_features, device_features, file_features, http_features, email_features, all_dfs
gc.collect()

print(f'\n✓ Fused daily behavioral matrix: {df_daily.shape}')
print(f'  Users: {df_daily["user"].nunique():,}')
print(f'  Days:  {df_daily["day"].nunique():,}')
print(f'  Features: {len(feature_cols)}')
print(f'  Names: {feature_cols}')
df_daily.head()

  Merging logon: (330452, 8)
  Merging device: (55717, 6)
  Merging file: (45907, 7)
  Merging http: (329845, 7)
  Merging email: (326985, 8)

✓ Fused daily behavioral matrix: (330452, 28)
  Users: 1,000
  Days:  501
  Features: 26
  Names: ['logon_total', 'logon_after_h', 'logon_weekend', 'logon_unique_pcs', 'logon_count', 'logoff_count', 'device_total', 'device_after_h', 'device_weekend', 'device_connects', 'file_total', 'file_after_h', 'file_weekend', 'file_unique_names', 'file_sensitive', 'http_total', 'http_after_h', 'http_weekend', 'http_unique_domains', 'http_suspicious', 'email_total', 'email_after_h', 'email_weekend', 'email_attachments', 'email_avg_size', 'email_recipients']
CPU times: user 787 ms, sys: 96 ms, total: 883 ms
Wall time: 853 ms


,user,day,logon_total,logon_after_h,logon_weekend,logon_unique_pcs,logon_count,logoff_count,device_total,device_after_h,...,http_after_h,http_weekend,http_unique_domains,http_suspicious,email_total,email_after_h,email_weekend,email_attachments,email_avg_size,email_recipients
0,AAE0190,2010-01-04,2.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,7.0,0.0,54.0,1.0,14.0,0.0,0.0,4.0,31523.427734,24.0
1,AAE0190,2010-01-05,2.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,53.0,3.0,13.0,0.0,0.0,2.0,27350.154297,24.0
2,AAE0190,2010-01-06,2.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,3.0,0.0,51.0,7.0,14.0,0.0,0.0,12.0,38046.214844,21.0
3,AAE0190,2010-01-07,2.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,8.0,0.0,48.0,8.0,14.0,0.0,0.0,9.0,33902.214844,21.0
4,AAE0190,2010-01-08,2.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,3.0,0.0,53.0,2.0,13.0,0.0,0.0,10.0,30818.230469,21.0


## Step 6 — Fuse All Sources into Daily Behavioral Vectors

We merge all per-user-per-day feature tables into a single unified behavioral
profile. Missing sources get zero-filled (a user with no USB activity on a day
gets `device_total_events=0`).

In [12]:
%%time

def assign_label(row):
    uid = str(row['user']).strip()
    if uid not in insider_users:
        return 'normal'
    day = row['day']
    for ev in insider_users[uid]:
        s, e = ev['start'], ev['end']
        if s is None and e is None: return 'insider_threat'
        if s is not None and e is None and day >= s: return 'insider_threat'
        if s is not None and e is not None and s <= day <= e: return 'insider_threat'
    return 'normal'

df_daily['label'] = df_daily.apply(assign_label, axis=1)
print('Label distribution (user-day):')
print(df_daily['label'].value_counts())

n_threat = (df_daily['label'] == 'insider_threat').sum()
if n_threat < 50 and len(insider_users) > 0:
    print(f'\n[INFO] Only {n_threat} threat days. Expanding to all insider user days...')
    df_daily['label'] = df_daily['user'].apply(
        lambda u: 'insider_threat' if str(u).strip() in insider_users else 'normal'
    )
    print(df_daily['label'].value_counts())

print(f'\nInsider ratio: {(df_daily["label"]=="insider_threat").mean():.4%}')

Label distribution (user-day):
label
normal            329158
insider_threat      1294
Name: count, dtype: int64

Insider ratio: 0.3916%
CPU times: user 1.91 s, sys: 111 ms, total: 2.03 s
Wall time: 2.02 s


## Step 7 — Label Assignment & Insider Threat Classification

We label each user-day as:
- `normal` — regular employee activity
- `insider_threat` — the user is a known insider AND this day falls within
  their active malicious period

In [13]:
%%time

scaler = StandardScaler()
df_daily[feature_cols] = scaler.fit_transform(df_daily[feature_cols])
joblib.dump(scaler, OUTPUT_PATH + 'scaler.pkl')

windows = []
window_labels = []
window_meta = []

print(f'Building {WINDOW_SIZE}-day sliding windows...')
for user_id, udf in df_daily.groupby('user'):
    udf = udf.sort_values('day').reset_index(drop=True)
    feats  = udf[feature_cols].values.astype(np.float32)
    labels = udf['label'].values
    days   = udf['day'].values
    for i in range(len(udf) - WINDOW_SIZE + 1):
        w_feats  = feats[i : i + WINDOW_SIZE]
        w_labels = labels[i : i + WINDOW_SIZE]
        w_label  = 'insider_threat' if 'insider_threat' in w_labels else 'normal'
        windows.append(w_feats)
        window_labels.append(w_label)
        window_meta.append({'user': str(user_id),
                            'start': str(days[i])[:10],
                            'end': str(days[i+WINDOW_SIZE-1])[:10]})

X_all = np.array(windows, dtype=np.float32)
y_all_str = np.array(window_labels)

del windows, df_daily; gc.collect()

print(f'\n✓ Windows: {X_all.shape}')
unique, counts = np.unique(y_all_str, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {u}: {c:,} ({c/len(y_all_str):.2%})')

Building 7-day sliding windows...

✓ Windows: (324452, 7, 26)
  insider_threat: 1,509 (0.47%)
  normal: 322,943 (99.53%)
CPU times: user 4.22 s, sys: 64 ms, total: 4.28 s
Wall time: 4.28 s


## Step 8 — Build 7-Day Sliding Windows

We group the daily feature vectors into 7-day sliding windows per user.
Each window captures one week of behavioral evolution. The label for the
window is `insider_threat` if **any** day in the window is malicious.

In [ ]:
%%time

NUM_CLASSES = len(np.unique(y_all_str))
N_FEATURES  = X_all.shape[2]
master_le = LabelEncoder()
y_all_enc = master_le.fit_transform(y_all_str)
joblib.dump(master_le, OUTPUT_PATH + 'label_encoder.pkl')

print(f'Classes: {list(master_le.classes_)}')

# 3% inference holdout
inf_idx, rem_idx = [], []
rng = np.random.RandomState(42)
for ci in range(NUM_CLASSES):
    idxs = np.where(y_all_enc == ci)[0]
    n = max(int(len(idxs) * 0.03), 5)
    chosen = rng.choice(idxs, size=min(n, len(idxs)), replace=False)
    inf_idx.extend(chosen)
    rem_idx.extend([i for i in idxs if i not in chosen])

X_inf, y_inf = X_all[inf_idx], y_all_enc[inf_idx]
meta_inf = [window_meta[i] for i in inf_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_all[rem_idx], y_all_enc[rem_idx],
    test_size=0.20, random_state=42, stratify=y_all_enc[rem_idx]
)

del X_all, y_all_enc, y_all_str; gc.collect()
print(f'X_train:{X_train.shape}  X_test:{X_test.shape}  X_inf:{X_inf.shape}')

## Step 9 — Label Encoding & Train/Test Split

In [ ]:
def build_ueba_model(n_features, num_classes, window_size=WINDOW_SIZE):
    inputs = Input(shape=(window_size, n_features), name='input')
    x = BatchNormalization()(inputs)

    # Multi-scale CNN
    c3 = Conv1D(64, 3, padding='same', activation='gelu')(x)
    c5 = Conv1D(64, 5, padding='same', activation='gelu')(x)
    c7 = Conv1D(64, 7, padding='same', activation='gelu')(x)
    x = Concatenate()([c3, c5, c7])
    x = Conv1D(128, 1, padding='same', activation='gelu')(x)
    x = LayerNormalization()(x)

    # BiLSTM
    x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3))(x)

    # Multi-Head Attention
    attn = MultiHeadAttention(num_heads=8, key_dim=32)(x, x)
    x = Add()([x, attn])
    x = LayerNormalization()(x)

    # Classification
    x = GlobalAveragePooling1D()(x)
    x = Dense(256, activation='gelu')(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='gelu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)

    return Model(inputs, out, name='UEBA_CNN_BiLSTM_Attention')

model = build_ueba_model(N_FEATURES, NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
    loss=FocalLoss(alpha=0.25, gamma=2.0),
    metrics=['accuracy']
)
model.summary()

## Step 10 — Build Model: MultiScale CNN + BiLSTM + 8-Head Attention

Phase 2 of the architecture. Input shape: `(7, n_features)` — 7 timesteps
(days) × N behavioral features per day.

In [ ]:
%%time

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50, batch_size=256,
    callbacks=[
        CosineAnnealingWarmRestarts(initial_lr=0.001, T_0=10, T_mult=2),
        MacroF1EarlyStopping(X_val=X_test, y_val=y_test, patience=7)
    ],
    verbose=1
)

y_pred = np.argmax(model.predict(X_test, batch_size=512), axis=1)
print('\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=master_le.classes_, zero_division=0))

## Step 11 — Train & Evaluate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=master_le.classes_, yticklabels=master_le.classes_, ax=axes[0])
axes[0].set_title('Confusion Matrix'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Pred')

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Focal Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history.history['accuracy'], label='Train')
axes[2].plot(history.history['val_accuracy'], label='Val')
axes[2].set_title('Accuracy'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'training_curves.png', dpi=150)
plt.show()

## Step 12 — Confusion Matrix & Training Curves

In [ ]:
%%time

MITRE = {
    'insider_threat': {
        'tactic': 'Exfiltration', 'tactic_id': 'TA0010',
        'technique': 'Exfiltration Over Alternative Protocol', 'technique_id': 'T1048',
        'kill_chain_stage': 7, 'kill_chain_name': 'Actions on Objectives',
        'description': 'Insider threat — anomalous behavioral pattern across logon, device, file, web, email.'
    },
    'normal': {
        'tactic': None, 'tactic_id': None, 'technique': None, 'technique_id': None,
        'kill_chain_stage': 0, 'kill_chain_name': 'No Threat',
        'description': 'Normal employee behavior.'
    }
}

probs = model.predict(X_inf, batch_size=256, verbose=0)
preds = np.argmax(probs, axis=1)
confs = np.max(probs, axis=1)

results = []
for i in range(len(X_inf)):
    pc = master_le.classes_[preds[i]]
    tc = master_le.classes_[y_inf[i]]
    cf = round(float(confs[i]) * 100, 2)
    m  = meta_inf[i]
    uid = m['user']
    is_t = pc != 'normal'

    results.append({
        'id': f'UEBA-{uid}-{m["start"]}', 'block_id': f'UEBA-{uid}-{m["start"]}',
        'session_key': uid, 'source': SOURCE_NAME, 'user_id': uid,
        'window_start': m['start'], 'window_end': m['end'],
        'prediction': 'Insider Threat' if is_t else 'Normal',
        'verdict': 'THREAT' if is_t else 'NORMAL',
        'attack_type': pc if is_t else None,
        'confidence': cf, 'true_label': tc, 'is_correct': pc == tc,
        'preview': f'UEBA | User: {uid} | {m["start"]} - {m["end"]}',
        'label': 'Insider Threat' if is_t else 'Normal',
        'title': 'Insider Threat Detected' if is_t else 'Normal Activity',
        'severity': 'critical' if cf > 85 else ('warning' if is_t else 'info'),
        'time': datetime.datetime.utcnow().isoformat(),
        'mitre': MITRE.get(pc, MITRE['normal'])
    })

with open(OUTPUT_PATH + 'inference_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

model.save(OUTPUT_PATH + 'ueba_model.keras')
joblib.dump(feature_cols, OUTPUT_PATH + 'feature_cols.pkl')

meta = {'source': SOURCE_NAME, 'architecture': 'MultiScale_CNN_BiLSTM_8Head_Attention',
        'window_size': WINDOW_SIZE, 'n_features': N_FEATURES,
        'feature_names': feature_cols, 'classes': list(master_le.classes_),
        'dataset': 'CERT Insider Threat r4.2 (CMU SEI)'}
with open(OUTPUT_PATH + 'model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

correct = sum(1 for r in results if r['is_correct'])
print(f'\n✓ All saved to {OUTPUT_PATH}')
print(f'  Inference accuracy: {correct}/{len(results)} ({correct/len(results):.1%})')

## Step 13 — Save Model, Inference Results & Artifacts

In [ ]:
%%time

UEBA_MITRE_MAP = {
    'insider_threat': {
        'tactic': 'Exfiltration', 'tactic_id': 'TA0010',
        'technique': 'Exfiltration Over Alternative Protocol',
        'technique_id': 'T1048',
        'kill_chain_stage': 7, 'kill_chain_name': 'Actions on Objectives',
        'description': 'Insider threat detected — anomalous behavioral pattern '
                       'across logon, device, file, web, and email activity.'
    },
    'normal': {
        'tactic': None, 'tactic_id': None,
        'technique': None, 'technique_id': None,
        'kill_chain_stage': 0, 'kill_chain_name': 'No Threat',
        'description': 'Normal employee behavioral pattern.'
    }
}

# ── Run inference on holdout ──────────────────────────────────────────────────
inf_probs = model.predict(X_inf, batch_size=256, verbose=0)
inf_preds = np.argmax(inf_probs, axis=1)
inf_confs = np.max(inf_probs, axis=1)

results = []
for i in range(len(X_inf)):
    pred_class = master_le.classes_[inf_preds[i]]
    true_class = master_le.classes_[y_inf[i]]
    conf_pct   = round(float(inf_confs[i]) * 100, 2)
    meta       = meta_inf[i]
    user_id    = meta['user']
    w_start    = meta['window_start'][:10]
    alert_id   = f'UEBA-{user_id}-{w_start}'
    is_threat  = pred_class != 'normal'
    mitre      = UEBA_MITRE_MAP.get(pred_class, UEBA_MITRE_MAP['normal'])

    results.append({
        'id': alert_id,
        'block_id': alert_id,
        'session_key': user_id,
        'source': SOURCE_NAME,
        'user_id': user_id,
        'window_start': meta['window_start'],
        'window_end':   meta['window_end'],
        'prediction': 'Insider Threat' if is_threat else 'Normal',
        'verdict': 'THREAT' if is_threat else 'NORMAL',
        'attack_type': pred_class if is_threat else None,
        'confidence': conf_pct,
        'true_label': true_class,
        'is_correct': pred_class == true_class,
        'preview': f'UEBA | User: {user_id} | {w_start} | 7-day window',
        'label': 'Insider Threat' if is_threat else 'Normal',
        'title': 'Insider Threat Detected' if is_threat else 'Normal User Activity',
        'severity': 'critical' if conf_pct > 85 else ('warning' if is_threat else 'info'),
        'time': datetime.datetime.utcnow().isoformat(),
        'mitre': mitre
    })

# ── Save everything ──────────────────────────────────────────────────────────
with open(OUTPUT_PATH + 'inference_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

model.save(OUTPUT_PATH + 'ueba_model.keras')
joblib.dump(feature_cols, OUTPUT_PATH + 'feature_cols.pkl')

# Save metadata for backend integration
model_meta = {
    'source': SOURCE_NAME,
    'architecture': 'MultiScale_CNN_BiLSTM_8Head_Attention',
    'window_size': WINDOW_SIZE,
    'n_features': N_FEATURES,
    'feature_names': feature_cols,
    'classes': list(master_le.classes_),
    'dataset': 'CERT Insider Threat r4.2 (CMU SEI)',
    'training_date': datetime.datetime.utcnow().isoformat()
}
with open(OUTPUT_PATH + 'model_metadata.json', 'w') as f:
    json.dump(model_meta, f, indent=2)

print('\n✓ All artifacts saved to', OUTPUT_PATH)
print(f'  ueba_model.keras')
print(f'  label_encoder.pkl')
print(f'  scaler.pkl')
print(f'  feature_cols.pkl')
print(f'  inference_results.json ({len(results)} samples)')
print(f'  model_metadata.json')

# ── Quick inference accuracy ─────────────────────────────────────────────────
correct = sum(1 for r in results if r['is_correct'])
print(f'\n  Inference holdout accuracy: {correct}/{len(results)} ({correct/len(results):.1%})')